In [1]:
import pandas as pd

In [2]:
df_sum_feb = pd.read_csv("../data/raw/listings_summary_feb2026.csv")
df_sum_apr = pd.read_csv("../data/raw/listings_summary_apr2026.csv")
df_feb = pd.read_csv("../data/raw/listings_feb2026.csv.gz")
df_apr = pd.read_csv("../data/raw/listings_apr2026.csv.gz")

In [3]:
cols_to_drop = [
    # 100% null
    'neighborhood_overview', 'host_since', 'host_response_time',
    'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url',
    'host_neighbourhood', 'host_total_listings_count', 'host_verifications',
    'calendar_updated', 'estimated_revenue_l365d', 'requires_license',
    'license', 'instant_bookable', 'neighbourhood',
    'region_parent_parent_id', 'region_parent_parent_name',
    # metadata/URLs - not predictive
    'listing_url', 'picture_url', 'host_url', 'host_picture_url',
    'scrape_id', 'last_searched', 'last_scraped', 'source',
    'calendar_last_scraped',
    # Apr-only redundant price metadata
    'price_quote_checkin_date', 'price_quote_checkout_date',
    'price_quote_raw', 'price_quote_total_price'
]
# Drop only columns that exist in each df
df_feb.drop(columns=[c for c in cols_to_drop if c in df_feb.columns], inplace=True)
df_apr.drop(columns=[c for c in cols_to_drop if c in df_apr.columns], inplace=True)

In [4]:
df_apr.shape

(48951, 64)

In [5]:
df_feb.shape

(48692, 59)

In [16]:
df_apr['price'] = df_apr['price'].str.replace("[$,]","",regex=True).astype(float)

In [17]:
print("*"*10)
print(df_apr['price'].describe())
print("*"*10)
print(df_apr['price'].isnull().sum())
print("*"*10)
print(df_apr['price_quote_price_per_night'].describe())

**********
count     45107.000000
mean        381.293281
std        1086.262338
min           5.500000
25%         177.000000
50%         272.000000
75%         434.000000
max      182817.000000
Name: price, dtype: float64
**********
3844
**********
count     45097.000000
mean        381.324393
std        1086.379607
min           5.500000
25%         177.000000
50%         272.000000
75%         434.130000
max      182817.000000
Name: price_quote_price_per_night, dtype: float64


In [18]:
price_null_mask = df_apr['price'].isnull()
quote_null_mask = df_apr['price_quote_price_per_night'].isnull()
print("Both null:", (price_null_mask & quote_null_mask).sum())
print("Price null but quote available:", (price_null_mask & ~quote_null_mask).sum())

Both null: 3844
Price null but quote available: 0


In [19]:
print("Above $1000:", (df_apr['price'] > 1000).sum())
print("Above $2000:", (df_apr['price'] > 2000).sum())
print("Above $5000:", (df_apr['price'] > 5000).sum())
print("Zero price:", (df_apr['price'] == 0).sum())

Above $1000: 1932
Above $2000: 345
Above $5000: 52
Zero price: 0


In [20]:
print(df_apr['price'].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

0.01      62.000
0.05      96.000
0.25     177.000
0.50     272.000
0.75     434.000
0.95     943.901
0.99    1800.000
Name: price, dtype: float64


In [25]:
feb_ids = set(df_feb['id'])
apr_ids = set(df_apr['id'])
print("Listings in both:", len(feb_ids & apr_ids))
print("Only in Feb:", len(feb_ids - apr_ids))
print("Only in Apr:", len(apr_ids - feb_ids))

Listings in both: 45650
Only in Feb: 3042
Only in Apr: 3301
